# Lily 1.5b v0.3 Non-GGUF Model Inference — Google Colab
**Interactive Reasoning & Schema Auditing for `abhinav0231/Lily-1.5b-v0.3`**

This notebook loads the un-quantized 16-bit standalone model `abhinav0231/Lily-1.5b-v0.3` from Hugging Face, executes ChatML reasoning inference, parses `<think>` and `<answer>` tags, and performs programmatic schema compliance auditing.

## Cell 1 — Install Transformers & Dependencies

In [ ]:
# ==============================================================================
# Cell 1 — Install Hugging Face Transformers & Acceleration Libraries
# ==============================================================================
!pip install -q -U transformers accelerate sentencepiece huggingface_hub


## Cell 2 — Load Non-GGUF Model (`abhinav0231/Lily-1.5b-v0.3`)

In [ ]:
# ==============================================================================
# Cell 2 — Authentication & Load Model (abhinav0231/Lily-1.5b-v0.3)
# ==============================================================================
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
try:
    from huggingface_hub import login, get_token
except ImportError:
    from huggingface_hub import login, HfFolder
    get_token = HfFolder.get_token

# Retrieve HF Token from all possible sources
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN", "")
if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_TOKEN") or ""
    except Exception:
        pass

if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    cached_token = get_token()
    if cached_token:
        HF_TOKEN = cached_token

if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("✅ Authenticated with Hugging Face")
    except Exception as e:
        print(f"⚠️ Hugging Face authentication note: {e}")

# Target un-quantized 16-bit distilled model repository on Hugging Face Hub
REPO_ID = "abhinav0231/Lily-1.5b-v0.3"
print(f"Loading tokenizer and 16-bit model weights from: {REPO_ID} ...")

tokenizer = AutoTokenizer.from_pretrained(REPO_ID, trust_remote_code=True, token=HF_TOKEN if HF_TOKEN else None)
model = AutoModelForCausalLM.from_pretrained(
    REPO_ID,
    torch_dtype        = torch.bfloat16,
    device_map         = "auto",
    trust_remote_code  = True,
    token              = HF_TOKEN if HF_TOKEN else None,
)

print(f"✅ Model + Tokenizer loaded successfully on device: {model.device}")


## Cell 3 — Inference Function & Tag Parser

In [ ]:
# ==============================================================================
# Cell 3 — Prompt Formatting, Generation Loop & Reasoning CoT Parser
# ==============================================================================
import re

SYSTEM_PROMPT = (
    "You are a precise, helpful assistant. "
    "Always reason step by step inside <think></think> tags, "
    "then write your final answer inside <answer></answer> tags."
)

def ask(question, max_new_tokens=1024, temperature=0.7):
    """
    Applies standard ChatML template, generates completion via PyTorch, and decodes.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = "pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens = max_new_tokens,
            temperature    = temperature,
            top_p          = 0.95,
            do_sample      = True if temperature > 0 else False,
            pad_token_id   = tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        output_ids[0][input_ids.shape[-1]:],
        skip_special_tokens = True
    )
    return response

def parse_and_print(response):
    """
    Extracts and prints 'think_body' (<think>...</think>) and 'answer_body' (<answer>...</answer>).
    """
    think_m  = re.search(r"<think>(.*?)</think>", response, re.DOTALL)
    answer_m = re.search(r"<answer>(.*?)</answer>", response, re.DOTALL)

    print("🧠 REASONING (<think>):")
    if think_m:
        print(think_m.group(1).strip())
    else:
        print("No <think> tag found. Full output:")
        print(response)

    print("\n🎯 FINAL ANSWER (<answer>):")
    if answer_m:
        print(answer_m.group(1).strip())
    else:
        print(response.split("</think>")[-1].strip())

print("✅ Inference & Parsing functions ready")


## Cell 4 — Programmatic Schema Compliance Auditor

In [ ]:
# ==============================================================================
# Cell 4 — Programmatic Schema Compliance Engine & Multi-Domain Test Suite
# ==============================================================================
def extract_stats(text):
    """Audits structural integrity of think and answer tags."""
    return {
        "has_think_open":   "<think>" in text,
        "has_think_close":  "</think>" in text,
        "has_answer_open":  "<answer>" in text,
        "has_answer_close": "</answer>" in text,
        "think_open_cnt":   text.count("<think>"),
        "think_close_cnt":  text.count("</think>"),
        "answer_open_cnt":  text.count("<answer>"),
        "answer_close_cnt": text.count("</answer>"),
        "has_thinking_leak": ("<thinking>" in text) or ("</thinking>" in text),
    }

def classify_case(raw_stats, text):
    s = raw_stats
    if (s["think_open_cnt"] == 1 and s["think_close_cnt"] == 1 and
        s["answer_open_cnt"] == 1 and s["answer_close_cnt"] == 1 and
        not s["has_thinking_leak"]):
        return "PASS_clean"
    if s["has_thinking_leak"]:
        return "FAIL_thinking_leak"
    if s["think_open_cnt"] != s["think_close_cnt"] or s["answer_open_cnt"] != s["answer_close_cnt"]:
        return "FAIL_unbalanced_tags"
    return "FAIL_no_schema"

TEST_CASES = [
    ("Math (Percentage)", "What is 15% of 840?"),
    ("Math (Algebra)", "A shop sells pens for $3 and notebooks for $7. If Alex buys 12 items total and spends $52, how many pens did Alex buy?"),
    ("Logic (Syllogism)", "If all Bloops are Razzies and all Razzies are Lazzies, are all Bloops definitely Lazzies? Explain."),
    ("Physics (Speed)", "If a train travels 120 km in 1.5 hours, what is its speed in meters per second?"),
    ("Coding (Python)", "Write a Python function to check if a string is a palindrome ignoring spaces and punctuation."),
    ("Puzzle", "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?")
]

results = []
print(f"Running automated compliance test suite across {len(TEST_CASES)} diverse reasoning cases...\n")

for domain, q in TEST_CASES:
    print("=" * 80)
    print(f"📌 Domain: {domain}")
    print(f"❓ Question: {q}")
    print("=" * 80)
    raw_out = ask(q, temperature=0.7)
    parse_and_print(raw_out)
    
    stats = extract_stats(raw_out)
    cls_label = classify_case(stats, raw_out)
    results.append({"domain": domain, "classification": cls_label, "stats": stats})
    print(f"Status: {cls_label}\n")

# Summary Report
print("=" * 80)
print("📊 EVALUATION & SCHEMA COMPLIANCE SUMMARY REPORT")
print("=" * 80)
passes = sum(1 for r in results if r["classification"] == "PASS_clean")
print(f"Total Tests Run   : {len(results)}")
print(f"Clean Pass Rate   : {passes}/{len(results)} ({100 * passes / len(results):.1f}%)")
for r in results:
    print(f"  - [{r['domain']}]: {r['classification']}")


## Cell 5 — Interactive Custom Query Loop


In [ ]:
# ==============================================================================
# Cell 5 — Interactive User Query Loop
# ==============================================================================
print("Type your custom query below (or type 'exit' to quit):\n")
while True:
    user_query = input("Enter Query: ")
    if user_query.strip().lower() in ["exit", "quit", "q"]:
        print("Exiting interactive session.")
        break
    if not user_query.strip():
        continue

    print("\n" + "=" * 80)
    print(f"❓ QUESTION: {user_query}")
    print("=" * 80)
    raw_out = ask(user_query, temperature=0.7)
    parse_and_print(raw_out)
    print("\n")
